# DUET Classification Notebook

Этот блокнот приведён в соответствие с текущей архитектурой DUET для **классификации временных рядов**:
- **TCM** сегментирует ряд на патчи, выделяет тренд/сезонность и маршрутизирует их по temporal-кластерам.
- **CCM** работает в частотной области, учит межканальные связи и разреживает маску.
- **Fusion** объединяет temporal-признаки и маску каналов, после чего **classification head** выдаёт logits классов.

Ниже используются параметры `K_t`, `K_c`, `d_c`, `top_k`, а также RevIN/InstanceNorm в соответствии с `architecture.md`.


In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [1]:
import sys
sys.path.append("/content/drive/MyDrive/stocks/duet/duet_class")

from pipeline.config import DUETConfig
from pipeline import preprocess, train, evaluate, predict
from pipeline.prepare_and_check import FinancialTimeSeriesPreparer
from pipeline.wf_slicer import GlobalNormConfig, SplitConfig, WalkForwardWindowSlicerVec, WindowConfig
from duet.model import DUETModel
from torch.utils.data import DataLoader, TensorDataset
import torch
import joblib
import random

import numpy as np
import pandas as pd
from sklearn.utils.class_weight import compute_class_weight


In [2]:
# Настройки
DATA_PATH = "/content/drive/MyDrive/stocks/Data/anomaly/anomaly_df_binance_BTCUSDT_futures_cufOff_1.5.joblib"
LABELS_PATH = "/content/drive/MyDrive/stocks/Data/anomaly/Dataset_2_7class_clear - Dataset_2_7class_clear.csv.csv"
device = "cuda" if torch.cuda.is_available() else "cpu"

config = DUETConfig(
    # =========================
    # Общие параметры
    # =========================
    timestamp_col = "timestamp",
    features = [
       'Open', 'High', 'Low', 'Close',
       'Volume',
    ],                           # Название колонок для обучения
    forecast = 'anomaly_class',       # Название колонки с таргетом
    not_to_normalise = [],       # Название колонок, которые НЕ НАДО нормализовать
    scaler = 'NONE',         # Тип нормализации (none, STD, MINMAX, QUANT)
    predict_type = 'detect',     # Детекция "detect" или предикт "next" следующей свечи
    seq_len = 48,                # Длина входной последовательности
    num_classes = 7,             # кол-во классов
    patch_len = 24,               # Длина патча (TCM)
    stride = 12,                  # Шаг между патчами (TCM)
    moving_avg = 25,              # Размер окна скользящего сглаживания

    K_t = 2,                    # Кол-во временных кластеров (TCM)
    K_c = 4,                    # Кол-во кластеров каналов (CCM)
    d_c = 32,                   # Размерность embedding каналов (CCM)
    top_k = 2,                  # Кол-во связей при разреживании маски
    use_revin = True,           # Включить RevIN/InstanceNorm
    revin_affine = True,        # Использовать affine параметры в RevIN
    revin_eps = 1e-5,           # Эпсилон для стабильности RevIN

    # =========================
    # Параметры модели
    # =========================
    d_model = 64,               # Размерность скрытого пространства в attention
    d_ff = 256,                 # Размерность feedforward слоя
    n_heads = 4,                # Количество голов в multi-head attention
    e_layers = 2,               # Количество слоев в encoder (CCM)
    dropout = 0.2,            # Dropout во всех слоях attention
    fc_dropout = 0.2,         # Dropout в выходном head слое
    activation = "relu",        # Активационная функция (relu, gelu, elu)
    num_experts = 4,            # Число экспертов (в Router, если используется)
    report_freq = 10,            # Частота появления confusion matrix

    # =========================
    # Режимы обработки
    # =========================
    CI = True,                 # Channel-Independent режим (если False — shared weights)
    use_router = True,          # Включить распределительный роутер
    timeenc = 1,                # Использовать time encoding (0 = без, 1 = sin/cos и т.п.)

    # =========================
    # Настройки обучения
    # =========================
    batch_size=32,          # можно немного увеличить при хорошем GPU
    epochs=300,             # больше эпох для более тонкой настройки
    learning_rate=5e-4,     # стандартный LR для Adam
    weight_decay=1e-5,
    patience=300,            # early stopping не слишком строгий

    # =========================
    # Прочее
    # =========================
    checkpoint_best = '/content/drive/MyDrive/stocks/duet_class/weights/best_val_acc_weights_001.pt',       # Путь для сохранения лучших по val_accuracy весов
    checkpoint_final = '/content/drive/MyDrive/stocks/duet_class/weights/final_weights_001.pt',      # Путь для сохранения финальных весов
    seed = 11,                  # Фиксированное зерно генератора случайных чисел
    verbose = True,            # Печать хода обучения
)

"""
Устанавливает seed для numpy, random, torch (вкл. CUDA).
Гарантирует воспроизводимость.
"""

random.seed(config.seed)
np.random.seed(config.seed)
torch.manual_seed(config.seed)
torch.cuda.manual_seed_all(config.seed)

## Варианты запуска обучения (наборы конфигураций)

Ниже — типовые варианты запуска, которые отражают логику обучения DUET: end-to-end обучение **TCM → CCM → Fusion → Head** с кросс-энтропией. Для каждого варианта меняются только ключевые параметры архитектуры и кластеризации, остальные поля берутся из базовой конфигурации.

**1) Базовый режим (баланс точности и стабильности):**
```python
DUETConfig(
    K_t=2, K_c=4, d_c=32, top_k=2,
    use_router=True, use_revin=True,
    d_model=64, d_ff=128, e_layers=2,
)
```

**2) Больше временных режимов (сложная динамика):**
```python
DUETConfig(
    K_t=4, patch_len=12, stride=6,
    use_router=True, num_experts=6,
    d_model=96, d_ff=256,
)
```

**3) Усиленная кластеризация каналов (много каналов/шумные связи):**
```python
DUETConfig(
    K_c=6, d_c=48, top_k=3,
    e_layers=3, dropout=0.15,
)
```

**4) Ablation без temporal-router (проверка вклада маршрутизации):**
```python
DUETConfig(
    use_router=False, K_t=1,
    d_model=64, d_ff=128,
)
```

Для запуска достаточно заменить поля в блоке `config = DUETConfig(...)` ниже или создать отдельные конфиги и прогнать обучение в цикле.


In [3]:
# @title Загрузка данных

df = joblib.load(DATA_PATH)
labels = pd.read_csv(LABELS_PATH)

preparer = FinancialTimeSeriesPreparer(
    tz="UTC",
    timestamp_col="timestamp",
    drop_warmup=True,
)
labels_preparer = FinancialTimeSeriesPreparer(
    tz="UTC",
    timestamp_col="Time_close",
    drop_warmup=True,
)
df, _ = preparer.prepare(df, ensure_ohlcv=True, from_date='2023-01-01', to_date = "2024-01-19",)
labels, _ = labels_preparer.prepare(labels, ensure_ohlcv=False)
labels.index = labels.index.round("1s")     # до ближайшей секунды
labels.index = labels.index.round("1min")   # до ближайшей минуты



In [4]:
labels

,Unnamed: 0,Time_close,target,gaps,nan_filled,candle_error
datetime,,,,,,
2023-01-01 18:12:00+00:00,0.0,1.672597e+12,3.0,0,0,0
2023-01-01 18:13:00+00:00,0.0,1.672597e+12,NaN,1,1,0
2023-01-01 18:14:00+00:00,0.0,1.672597e+12,NaN,1,1,0
2023-01-01 18:15:00+00:00,0.0,1.672597e+12,NaN,1,1,0
2023-01-01 18:16:00+00:00,0.0,1.672597e+12,NaN,1,1,0
...,...,...,...,...,...,...
2024-01-18 15:15:00+00:00,2010.0,1.705591e+12,NaN,1,1,0
2024-01-18 15:16:00+00:00,2010.0,1.705591e+12,NaN,1,1,0
2024-01-18 15:17:00+00:00,2010.0,1.705591e+12,NaN,1,1,0


In [5]:
# Переносим колонку target, выравнивая по индексу
df['anomaly_class'] = labels['target']
df.fillna({'anomaly_class':-1}, inplace=True)


# Проверка
print("Размер объединенного df:", df.shape)
print("Колонки:", df.columns.tolist())
print("Заполненных target:", df['anomaly_class'].count())

Размер объединенного df: (551520, 43)
Колонки: ['timestamp', 'Open', 'High', 'Low', 'Close', 'Volume', 'Quote_asset_volume', 'Number_of_trades', 'Taker_buy_base_asset_volume', 'Taker_buy_quote_asset_volume', 'dPercent120_slide', 'dPercent15_slide', 'dPercent240_slide', 'dPercent30_slide', 'dPercent5_slide', 'dPercent60_slide', 'volDivNTradesDivMaxVol120_slide', 'volDivNTradesDivMaxVol15_slide', 'volDivNTradesDivMaxVol240_slide', 'volDivNTradesDivMaxVol30_slide', 'volDivNTradesDivMaxVol5_slide', 'volDivNTradesDivMaxVol60_slide', 'type_frame', 'dPercent120_slide_z', 'dPercent15_slide_z', 'dPercent240_slide_z', 'dPercent30_slide_z', 'dPercent5_slide_z', 'dPercent60_slide_z', 'volDivNTradesDivMaxVol120_slide_z', 'volDivNTradesDivMaxVol15_slide_z', 'volDivNTradesDivMaxVol240_slide_z', 'volDivNTradesDivMaxVol30_slide_z', 'volDivNTradesDivMaxVol5_slide_z', 'volDivNTradesDivMaxVol60_slide_z', 'zScore_mean', 'zScore_median', 'zScore_bin', 'iFrorest', 'gaps', 'nan_filled', 'candle_error', 'anoma

In [6]:
df['anomaly_class'].value_counts()

,count
anomaly_class,
-1.0,549511
1.0,825
3.0,629
0.0,173
2.0,166
4.0,136
5.0,50
6.0,30


In [15]:
# --- 1. Предобработка данных ---

split = SplitConfig(
    n_folds=1,
    mode="expanding",
    ratios=(0.70, 0.15, 0.15),
    step_size=None,
    gap=30,
    sliding_train_size=None,
)

if config.predict_type.upper() == "DETECT":
    y_end_offset = 0
elif config.predict_type.upper() == "NEXT":
    y_end_offset = 1
else:
    raise ValueError("config.predict_type должен быть 'DETECT' или 'NEXT'")

window = WindowConfig(
    x_window=config.seq_len,
    x_end_offset=0,
    y_window=1,
    y_end_offset=y_end_offset,
    allow_left_context_for_x=False,
)

scaler_map = {"STD": "standard", "MINMAX": "minmax", "QUANT": "quantile", "NONE": "none"}
global_norm = GlobalNormConfig(scaler=scaler_map.get(config.scaler.upper(), "none"))

slicer = WalkForwardWindowSlicerVec(
    split=split,
    window=window,
    global_norm=global_norm,
    no_norm_cols=config.not_to_normalise,
    eps=1e-12,
    drop_incomplete_last_fold=True,
)

out = slicer.split_and_window(
    X=df[config.features],
    y=df[config.forecast],
    aux=df[["gaps",	"nan_filled",	"candle_error"]]
)
fold0 = out["fold_0"]

x_train = fold0["train"]["X"]
y_train = fold0["train"]["y"][:, 0, 0].astype(int)
x_val = fold0["val"]["X"]
y_val = fold0["val"]["y"][:, 0, 0].astype(int)




In [16]:

# Балансируем (если не нужно, то закомментировать)
x_train, y_train = preprocess.balance_windows(x_train, y_train, max_ratio=100, exclude_labels=-1)
x_val, y_val = preprocess.balance_windows(x_val, y_val, max_ratio=100, exclude_labels=-1)

# Проверяем балансировку
unique, counts = np.unique(y_train, return_counts=True)
print("Классы после балансировки:", dict(zip(unique, counts)))

# --- 3. Подсчет весов классов ---
class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weights = torch.tensor(class_weights, dtype=torch.float32).to(device)
print(f"Распределение весов между класcами {class_weights}")


Балансировка: min=23, max=645, max_ratio=100.0 -> cap=2300
Балансировка: min=10, max=169, max_ratio=100.0 -> cap=1000
Классы после балансировки: {np.int64(0): np.int64(122), np.int64(1): np.int64(645), np.int64(2): np.int64(143), np.int64(3): np.int64(423), np.int64(4): np.int64(116), np.int64(5): np.int64(38), np.int64(6): np.int64(23)}
Распределение весов между класcами tensor([1.7681, 0.3344, 1.5085, 0.5100, 1.8596, 5.6767, 9.3789],
       device='cuda:0')


In [17]:

# --- 4. Создание DataLoader'ов ---
train_dataset = TensorDataset(
    torch.tensor(x_train, dtype=torch.float32),
    torch.tensor(y_train, dtype=torch.long)
)
val_dataset = TensorDataset(
    torch.tensor(x_val, dtype=torch.float32),
    torch.tensor(y_val, dtype=torch.long)
)

train_loader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=config.batch_size)

# --- 6. Инициализация модели ---
model = DUETModel(config).to(device)

# --- 7. Обучение модели ---
trained_model = train.train_model(
    model,
    config,
    train_loader,
    val_loader,
    device=device,
    class_weights = class_weights
)

RuntimeError: The size of tensor a (5) must match the size of tensor b (15) at non-singleton dimension 2

TypeError: 'DataLoader' object is not an iterator

In [ ]:
# Оценка модели
y_true, y_pred = evaluate.evaluate_model(model, val_loader, device="cuda")

# Подсчёт метрик
metrics = evaluate.compute_metrics(y_true, y_pred)
print("\nMetrics:")
for metric, value in metrics.items():
    print(f"{metric.upper()}: {value:.2f}")


# Матрица ошибок
evaluate.plot_confusion_matrix(y_true, y_pred)

# Примеры предсказаний
evaluate.plot_classification_examples(y_true, y_pred, n=10)

In [ ]:
# Загрузка весов

model.load_state_dict(torch.load(config.checkpoint_best)) # загрузка лучших весов
# model.load_state_dict(torch.load(config.checkpoint_final)) # загрузка финальных весов
model.eval()

In [ ]:
from pipeline.visualiser import plot_classification_forecast, plot_roc_auc, plot_pr_auc
from pipeline.predict import predict_dataset_batched

# Предсказания
df_val['pred_pivots'], y_probs = predict_dataset_batched(model, df_val, config, device="cuda")
# print(df_val)
# Выбор валидных индексов, где были предсказания
valid_mask = ~df_val['pred_pivots'].isna()

# Выравнивание по индексу
y_true = df_val.loc[valid_mask, config.forecast].values.astype(int)
y_pred_proba = y_probs[valid_mask.values]  # valid_mask должен быть ndarray такой же длины

# ROC-AUC
plot_roc_auc(y_true, y_pred_proba, n_classes=config.num_classes)

# PR-AUC
plot_pr_auc(y_true, y_pred_proba, n_classes=config.num_classes)

# Диагностика предсказаний
df_val.dropna(inplace=True)
print(df_val['pred_pivots'].value_counts(dropna=False))


In [ ]:
plot_classification_forecast(
    df_val,
    price_col="Close",
    target_col="pivots",
    forecast_col="pred_pivots",
    class_labels=["Valley", "Neutral", "Peak"],
    start=500,
    length=300,
    title="Classification Forecast: Price and Predicted Regimes"
)